# Lesson 5.3 — Best Practices & Linting

**Phase 5: Real-World Workflows**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Apply Helm chart best practices from the official guide
- Use `helm lint` and `helm template | kubectl apply --dry-run` as quality gates
- Structure values, labels, and annotations correctly
- Avoid the most common Helm anti-patterns

---

## Official Best Practices Reference

> [Helm Chart Best Practices](https://helm.sh/docs/chart_best_practices/) — https://helm.sh/docs/chart_best_practices/

---

## 1. Labels & Annotations

Use the **standard Kubernetes recommended labels** on every resource:

In [ ]:
%%writefile /tmp/labels-example.yaml
# These are the standard labels from https://kubernetes.io/docs/concepts/overview/working-with-objects/common-labels/
# and Helm best practices.

metadata:
  labels:
    # REQUIRED in every resource
    app.kubernetes.io/name: myapp          # Chart name
    app.kubernetes.io/instance: my-release # Release name — distinguishes two installs of same chart
    
    # RECOMMENDED
    app.kubernetes.io/version: "1.2.3"     # App version (Chart.AppVersion)
    app.kubernetes.io/component: backend   # Component role
    app.kubernetes.io/part-of: my-platform # Larger system this belongs to
    app.kubernetes.io/managed-by: Helm     # Always "Helm" for Helm-managed resources
    
    # Chart version — goes in annotations (not labels) because it changes on every release
  annotations:
    helm.sh/chart: myapp-0.3.0             # name + version of the chart

In [ ]:
cat /tmp/labels-example.yaml

> **Why `app.kubernetes.io/instance`?** It lets you deploy the same chart twice in the same namespace (e.g., `backend-v1` and `backend-v2`), and select the right pods with `kubectl get pods -l app.kubernetes.io/instance=my-release`.

---

## 2. Values Best Practices

In [ ]:
%%writefile /tmp/values-best-practices.yaml
# DO: Flat top-level keys for simple settings
replicaCount: 1
nameOverride: ""
fullnameOverride: ""

# DO: Group related settings into nested objects
image:
  repository: nginx
  pullPolicy: IfNotPresent
  tag: ""  # Defaults to appVersion when empty

# DO: Keep defaults as simple as possible — chart should work with zero customization
service:
  type: ClusterIP
  port: 80

# DO: Use 'enabled' booleans for optional components
ingress:
  enabled: false

serviceAccount:
  create: true

# DO: Accept empty maps for user-supplied metadata
podAnnotations: {}
podLabels: {}
nodeSelector: {}
tolerations: []
affinity: {}

# DON'T: Put secrets here
# password: supersecret   # WRONG — use existingSecret pattern instead
existingSecret: ""         # Reference a pre-created Kubernetes Secret

---

## 3. Template Best Practices

### Always Use `nindent` with `toYaml`

In [ ]:
cat << 'EOF'
# WRONG — can break YAML structure if tolerations is not empty
tolerations: {{ .Values.tolerations }}

# RIGHT — always safe, even when values is an empty list
tolerations:
  {{- toYaml .Values.tolerations | nindent 2 }}

# ALSO RIGHT — skip the block if empty
{{- if .Values.tolerations }}
tolerations:
  {{- toYaml .Values.tolerations | nindent 2 }}
{{- end }}
EOF

### Selector Labels Must Be Immutable

Selector labels on Deployments cannot be changed after creation. **Never** include the chart version in selectors:

In [ ]:
cat << 'EOF'
# WRONG — helm.sh/chart includes the version and will break on upgrade
selector:
  matchLabels:
    app.kubernetes.io/name: myapp
    helm.sh/chart: myapp-0.1.0   # VERSION CHANGES EVERY RELEASE — THIS WILL BREAK

# RIGHT — selector labels must be stable
selector:
  matchLabels:
    app.kubernetes.io/name: myapp
    app.kubernetes.io/instance: my-release
# These two never change — safe to use as selectors
EOF

---

## 4. Linting Your Chart

In [ ]:
# Create a chart with some deliberate issues to demonstrate lint
rm -rf /tmp/lintdemo && helm create /tmp/lintdemo

# Helm lint — catches structural, YAML, and schema errors
helm lint /tmp/lintdemo

echo ""
echo "== Lint with strict mode ==" 
helm lint /tmp/lintdemo --strict

In [ ]:
# Lint with a specific values file — catches value-dependent issues
cat > /tmp/custom-lint-values.yaml << 'EOF'
ingress:
  enabled: true
  className: nginx
  hosts:
    - host: myapp.example.com
      paths:
        - path: /
          pathType: Prefix
EOF

helm lint /tmp/lintdemo -f /tmp/custom-lint-values.yaml

---

## 5. The Full Validation Pipeline

In CI, run these checks in order:

In [ ]:
CHART=/tmp/lintdemo

echo "=== Step 1: Lint ==="
helm lint ${CHART} --strict

echo ""
echo "=== Step 2: Template render (offline) ==="
helm template my-release ${CHART} > /dev/null && echo "Templates rendered successfully"

echo ""
echo "=== Step 3: Kubectl dry-run (requires cluster) ==="
echo "# helm template my-release ${CHART} | kubectl apply --dry-run=client -f -"
echo "# This catches Kubernetes API validation errors (wrong apiVersion, missing fields, etc.)"
echo "# Run this if you have kubectl configured"

---

## 6. Common Anti-Patterns to Avoid

| Anti-Pattern | Problem | Fix |
|---|---|---|
| Secrets in `values.yaml` | Stored in plaintext in Git | Use `existingSecret` pattern |
| Hardcoded namespaces in templates | Breaks multi-namespace deployments | Use `{{ .Release.Namespace }}` |
| No resource limits | Pods can consume all cluster resources | Always define `resources.limits` |
| Chart version = app version | Confuses chart changes with app changes | Keep them separate in `Chart.yaml` |
| Selector labels with chart version | Breaks upgrades (immutable field) | Only use `name` + `instance` in selectors |
| No `NOTES.txt` | Users don't know how to use the release | Always write a helpful `NOTES.txt` |
| Unbounded lists with `--set` | Complex list syntax is error-prone | Prefer `-f values.yaml` for lists |

---

## Exercise

In [ ]:
# TASK 1: Create a chart that has AT LEAST one lint warning or error.
#         (Hint: try setting an invalid apiVersion, or an empty Chart.yaml field)
#         Then fix it and verify helm lint passes cleanly.
# TODO:


# TASK 2: Update the lintdemo _helpers.tpl to follow the
#         full standard label set (name, instance, version, managed-by).
#         Verify the rendered templates contain all labels.
# TODO:


# TASK 3: Create a values.yaml that uses 'existingSecret' instead of 'password'.
#         Write a template that creates an env var DATABASE_PASSWORD from
#         the secret IF existingSecret is provided.
# TODO:

echo "Exercise: implement the tasks above"

---

## Summary

- Use `app.kubernetes.io/*` labels on every resource; never put chart version in selectors.
- Put secrets only as references (`existingSecret`), never as plaintext in `values.yaml`.
- `helm lint --strict` + `helm template` + `kubectl apply --dry-run` = full CI quality gate.
- Charts should work with zero user config — sensible defaults in `values.yaml`.
- Always write a helpful `NOTES.txt` with post-install instructions.

---

## Next Lesson

➡️ [Lesson 5.4 — CI/CD Integration](lesson-5.4-cicd.ipynb)